# 📊 Giai đoạn 1: Khám phá dữ liệu & Trích xuất đặc trưng

Phần bàn giao của **TV2 – Feature Engineering & EDA**. Notebook sử dụng dữ liệu sạch do TV1 bàn giao, xuất biểu đồ vào `reports/figures/`, và tạo bộ đặc trưng cho TV3.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'reviews_cleaned.xlsx'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style='whitegrid', context='notebook')
df = pd.read_excel(DATA_PATH)
required_columns = {'Rating', 'sentiment', 'clean_advance_text'}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing)}')
print(f'Dữ liệu: {df.shape[0]:,} mẫu, {df.shape[1]} cột')
display(df.head(3))

## 1. Kiểm tra chất lượng và cấu trúc dữ liệu

In [ ]:
quality = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(dropna=True),
})
display(quality)
print(f'Số dòng trùng hoàn toàn: {df.duplicated().sum():,}')

## 2. Phân bố điểm đánh giá và nhãn cảm xúc

In [ ]:
rating_counts = df['Rating'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=rating_counts.index, y=rating_counts.values, color='#4C78A8', ax=ax)
ax.set(title='Phân bố điểm đánh giá ITviec', xlabel='Số sao', ylabel='Số lượng review')
for patch, value in zip(ax.patches, rating_counts.values):
    ax.annotate(f'{value:,}', (patch.get_x() + patch.get_width() / 2, value), ha='center', va='bottom')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_rating_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
display(rating_counts.rename('count').to_frame())

In [ ]:
sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_counts = df['sentiment'].value_counts().reindex(sentiment_order)
sentiment_pct = (sentiment_counts / len(df) * 100).round(2)
sentiment_summary = pd.DataFrame({'count': sentiment_counts, 'percent': sentiment_pct})
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ['#59A14F', '#F28E2B', '#E15759']
sns.barplot(x=sentiment_summary.index, y=sentiment_summary['count'], palette=colors, hue=sentiment_summary.index, legend=False, ax=ax)
ax.set(title='Phân bố nhãn cảm xúc', xlabel='Nhãn cảm xúc', ylabel='Số lượng review')
for patch, (count, percent) in zip(ax.patches, sentiment_summary[['count', 'percent']].itertuples(index=False, name=None)):
    ax.annotate(f'{count:,} ({percent:.2f}%)', (patch.get_x() + patch.get_width() / 2, count), ha='center', va='bottom')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_sentiment_counts.png', dpi=300, bbox_inches='tight')
plt.show()
display(sentiment_summary)

## 3. Độ dài phần khen và đề xuất cải thiện

Độ dài được tính theo số token phân tách bằng khoảng trắng. Trục biểu đồ được giới hạn tại phân vị 99 để các ngoại lệ rất dài không che khuất phân bố chính.

In [ ]:
length_columns = ['What I liked', 'Suggestions for improvement']
length_data = pd.DataFrame({column: df[column].fillna('').astype(str).str.split().str.len() for column in length_columns})
length_summary = length_data.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T.round(2)
display(length_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, column, color in zip(axes, length_columns, ['#4C78A8', '#E15759']):
    upper = max(1, int(length_data[column].quantile(0.99)))
    sns.histplot(length_data[column], bins=40, kde=True, color=color, ax=ax)
    ax.set_xlim(0, upper)
    ax.set(title=column, xlabel='Số từ', ylabel='Số lượng review')
fig.suptitle('Phân bố độ dài nội dung review (đến phân vị 99)')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_text_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Điểm khía cạnh và cảm xúc tổng thể

Dùng hệ số Spearman vì các điểm đánh giá là thang thứ bậc 1–5. `Rating` chỉ phục vụ EDA và tạo nhãn; **không** được đưa vào ma trận huấn luyện để tránh rò rỉ nhãn.

In [ ]:
aspect_columns = [
    'Salary & benefits',
    'Training & learning',
    'Management cares about me',
    'Culture & fun',
    'Office & workspace',
]
aspect_labels = {
    'Salary & benefits': 'Lương & phúc lợi',
    'Training & learning': 'Đào tạo',
    'Management cares about me': 'Quản lý',
    'Culture & fun': 'Văn hóa',
    'Office & workspace': 'Văn phòng',
}
correlations = df[aspect_columns + ['Rating']].corr(method='spearman')
correlations_display = correlations.rename(index=aspect_labels, columns=aspect_labels)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(correlations_display, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Tương quan Spearman giữa điểm khía cạnh và Rating')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_aspect_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

aspect_by_sentiment = df.groupby('sentiment')[aspect_columns].mean().reindex(sentiment_order)
display(aspect_by_sentiment.round(2))
fig, ax = plt.subplots(figsize=(11, 5))
aspect_by_sentiment.rename(columns=aspect_labels).T.plot(kind='bar', ax=ax, color=colors)
ax.set(title='Điểm khía cạnh trung bình theo cảm xúc', xlabel='Khía cạnh', ylabel='Điểm trung bình', ylim=(1, 5))
ax.legend(title='Cảm xúc')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_aspect_by_sentiment.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Mức tập trung theo công ty và biến động theo thời gian

EDA này cho biết dữ liệu có bị một vài công ty chi phối hay không và phân bố sentiment có thay đổi theo năm hay không.

In [ ]:
company_counts = df['Company Name'].value_counts()
df['review_date'] = pd.to_datetime(df['Cmt_day'], format='%B %Y', errors='coerce')
df['review_year'] = df['review_date'].dt.year
print(f'Số công ty: {company_counts.size}; công ty dưới 20 review: {(company_counts < 20).sum()}')
display(company_counts.head(15).rename('reviews').to_frame())

year_sentiment = pd.crosstab(df['review_year'], df['sentiment'], normalize='index').reindex(columns=sentiment_order) * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=company_counts.head(15).values, y=company_counts.head(15).index, color='#4C78A8', ax=axes[0])
axes[0].set(title='15 công ty có nhiều review nhất', xlabel='Số review', ylabel='Công ty')
year_sentiment.plot(ax=axes[1], color=colors, marker='o')
axes[1].set(title='Tỷ lệ sentiment theo năm', xlabel='Năm', ylabel='Tỷ lệ (%)', ylim=(0, 100))
axes[1].legend(title='Sentiment')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_company_time_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Chất lượng weak labels và coverage của lexicon

`sentiment` được suy ra từ Rating nên được xem là weak label. `Recommend?`, lexicon coverage và text trùng được dùng như tín hiệu audit, không phải bằng chứng biến weak label thành ground truth.

In [ ]:
recommend_audit = pd.crosstab(df['sentiment'], df['Recommend?']).reindex(sentiment_order)
lexicon_coverage = (df['total_we'] > 0).mean() * 100
emoji_nonzero = int(((df['pos_e'] > 0) | (df['neg_e'] > 0)).sum())
duplicate_rows = int(df.duplicated('clean_advance_text', keep=False).sum())
conflicting_text_groups = int((df.groupby('clean_advance_text')['sentiment'].nunique() > 1).sum())
length_by_sentiment = df.assign(words=df['raw_review_text'].fillna('').str.split().str.len()).groupby('sentiment')['words'].mean().reindex(sentiment_order)
print(f'Lexicon có tín hiệu: {lexicon_coverage:.2f}%')
print(f'Dòng có emoji feature khác 0: {emoji_nonzero:,}')
print(f'Dòng thuộc nhóm text trùng: {duplicate_rows}; nhóm text trùng khác nhãn: {conflicting_text_groups}')
display(recommend_audit)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(x=length_by_sentiment.index, y=length_by_sentiment.values, palette=colors, hue=length_by_sentiment.index, legend=False, ax=axes[0])
axes[0].set(title='Độ dài review trung bình theo weak label', xlabel='Weak label', ylabel='Số từ trung bình')
recommend_pct = recommend_audit.div(recommend_audit.sum(axis=1), axis=0) * 100
sns.heatmap(recommend_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1])
axes[1].set(title='Recommend? theo weak label (%)', xlabel='Recommend?', ylabel='Weak label')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_label_quality_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. So sánh cấu hình TF-IDF unigram và unigram+bigram

Hai cấu hình được chọn bằng Stratified K-Fold trên tập development. Final test được khóa riêng và không được dùng trong EDA/feature selection. Macro F1 được dùng vì dữ liệu mất cân bằng.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from src.features import deduplicate_modeling_rows

modeling_df, duplicate_audit = deduplicate_modeling_rows(df)
development_idx, final_test_idx = train_test_split(
    modeling_df.index, test_size=0.2, random_state=2026,
    stratify=modeling_df['sentiment'],
)
development_df = modeling_df.loc[development_idx]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
candidate_ngram_ranges = [(1, 1), (1, 2)]
ngram_results = []
for ngram_range in candidate_ngram_ranges:
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=ngram_range, min_df=2, sublinear_tf=True)),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=2026)),
    ])
    scores = cross_val_score(
        pipeline, development_df['clean_advance_text'].fillna(''),
        development_df['sentiment'], cv=cv, scoring='f1_macro', n_jobs=-1,
    )
    ngram_results.append({
        'ngram_range': str(ngram_range),
        'cv_macro_f1_mean': scores.mean(),
        'cv_macro_f1_std': scores.std(),
    })
ngram_results = pd.DataFrame(ngram_results)
selected_ngram = candidate_ngram_ranges[int(ngram_results['cv_macro_f1_mean'].idxmax())]
print(f'Cấu hình được chọn trên development CV: {selected_ngram}')
display(ngram_results.round(4))
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=ngram_results, x='ngram_range', y='cv_macro_f1_mean', color='#4C78A8', ax=ax)
ax.errorbar(range(len(ngram_results)), ngram_results['cv_macro_f1_mean'], yerr=ngram_results['cv_macro_f1_std'], fmt='none', color='black', capsize=4)
for patch, value in zip(ax.patches, ngram_results['cv_macro_f1_mean']):
    ax.annotate(f'{value:.3f}', (patch.get_x() + patch.get_width() / 2, value), ha='center', va='bottom')
ax.set(title='So sánh TF-IDF bằng 5-fold CV trên development', xlabel='N-gram range', ylabel='CV Macro F1', ylim=(0, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_tfidf_ngram_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Ablation: mô hình đang học từ text hay từ điểm số?

Các nhóm feature được so sánh bằng cùng 5-fold CV trên development. Điểm khía cạnh là thí nghiệm structured/tabular, không phải input mặc định của ứng dụng text-only.

In [ ]:
from scipy import sparse
from sklearn.preprocessing import MinMaxScaler
from src.features import ASPECT_RATING_FEATURES, LEXICON_FEATURES

ablation_scores = {name: [] for name in ['Text-only', 'Text + lexicon', 'Aspect ratings only', 'Full structured hybrid']}
for fold_train_pos, fold_valid_pos in cv.split(development_df, development_df['sentiment']):
    fold_train = development_df.iloc[fold_train_pos]
    fold_valid = development_df.iloc[fold_valid_pos]
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=selected_ngram, min_df=2, sublinear_tf=True)
    X_text_train = vectorizer.fit_transform(fold_train['clean_advance_text'].fillna(''))
    X_text_valid = vectorizer.transform(fold_valid['clean_advance_text'].fillna(''))

    lex_scaler = MinMaxScaler()
    X_lex_train = sparse.csr_matrix(lex_scaler.fit_transform(fold_train[list(LEXICON_FEATURES)]))
    X_lex_valid = sparse.csr_matrix(lex_scaler.transform(fold_valid[list(LEXICON_FEATURES)]))
    aspect_scaler = MinMaxScaler()
    X_aspect_train = sparse.csr_matrix(aspect_scaler.fit_transform(fold_train[list(ASPECT_RATING_FEATURES)]))
    X_aspect_valid = sparse.csr_matrix(aspect_scaler.transform(fold_valid[list(ASPECT_RATING_FEATURES)]))

    feature_sets = {
        'Text-only': (X_text_train, X_text_valid),
        'Text + lexicon': (sparse.hstack([X_text_train, X_lex_train], format='csr'), sparse.hstack([X_text_valid, X_lex_valid], format='csr')),
        'Aspect ratings only': (X_aspect_train, X_aspect_valid),
        'Full structured hybrid': (sparse.hstack([X_text_train, X_lex_train, X_aspect_train], format='csr'), sparse.hstack([X_text_valid, X_lex_valid, X_aspect_valid], format='csr')),
    }
    for name, (X_fold_train, X_fold_valid) in feature_sets.items():
        model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=2026)
        model.fit(X_fold_train, fold_train['sentiment'])
        prediction = model.predict(X_fold_valid)
        ablation_scores[name].append(f1_score(fold_valid['sentiment'], prediction, average='macro'))

ablation_results = pd.DataFrame([
    {'feature_group': name, 'cv_macro_f1_mean': np.mean(scores), 'cv_macro_f1_std': np.std(scores)}
    for name, scores in ablation_scores.items()
]).sort_values('cv_macro_f1_mean', ascending=False)
display(ablation_results.round(4))
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=ablation_results, x='cv_macro_f1_mean', y='feature_group', color='#4C78A8', ax=ax)
ax.errorbar(ablation_results['cv_macro_f1_mean'], range(len(ablation_results)), xerr=ablation_results['cv_macro_f1_std'], fmt='none', color='black', capsize=4)
for patch, value in zip(ax.patches, ablation_results['cv_macro_f1_mean']):
    ax.annotate(f'{value:.3f}', (value, patch.get_y() + patch.get_height() / 2), va='center', ha='left', xytext=(4, 0), textcoords='offset points')
ax.set(title='Ablation feature bằng 5-fold CV trên development', xlabel='CV Macro F1', ylabel='Nhóm feature', xlim=(0, 1))
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'eda_feature_ablation_cv.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Tạo artifact text-only và khóa final test

Artifact chính chỉ gồm TF-IDF text để khớp ứng dụng nhập văn bản. Final test dùng seed 2026, được khóa cho TV3 và không được TV2 dùng để chọn feature hay báo cáo điểm mô hình.

In [ ]:
import hashlib
import importlib.metadata as package_metadata
import json
import platform
import subprocess

from src.features import FeatureExtractor, prepare_feature_split, save_feature_split

text_extractor = FeatureExtractor(
    method='tfidf', max_features=5000, ngram_range=selected_ngram,
    min_df=2, numeric_features=(),
)
split = prepare_feature_split(
    modeling_df, extractor=text_extractor, numeric_columns=(),
    test_size=0.2, random_state=2026,
)
assert set(split.test_indices) == set(final_test_idx)

vectorizer_path = MODELS_DIR / 'text_tfidf_vectorizer.joblib'
extractor_path = MODELS_DIR / 'text_feature_extractor.joblib'
split_path = MODELS_DIR / 'train_test_features.joblib'
manifest_path = MODELS_DIR / 'artifact_manifest.json'
text_extractor.save_vectorizer(vectorizer_path)
text_extractor.save_bundle(extractor_path)
artifact_metadata = {
    'feature_mode': 'text_only',
    'text_column': 'clean_advance_text',
    'label_column': 'sentiment',
    'split_seed': 2026,
    'test_size': 0.2,
    'ngram_range': list(selected_ngram),
    'max_features': 5000,
    'min_df': 2,
    'final_test_policy': 'locked_not_evaluated_by_tv2',
    'duplicate_audit': duplicate_audit,
}
save_feature_split(split, split_path, extractor=text_extractor, metadata=artifact_metadata)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

artifact_paths = [vectorizer_path, extractor_path, split_path]
manifest = {
    'schema_version': 1,
    'git_sha': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip(),
    'git_branch': subprocess.check_output(['git', 'branch', '--show-current'], cwd=PROJECT_ROOT, text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip()),
    'data_file': str(DATA_PATH.relative_to(PROJECT_ROOT)),
    'data_sha256': sha256_file(DATA_PATH),
    'requirements_lock_sha256': sha256_file(PROJECT_ROOT / 'requirements.lock'),
    'source_rows': len(df),
    'modeling_rows': len(modeling_df),
    'train_rows': split.X_train.shape[0],
    'test_rows': split.X_test.shape[0],
    'feature_count': split.X_train.shape[1],
    'runtime': {
        'python': platform.python_version(),
        **{name: package_metadata.version(name) for name in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'joblib']},
    },
    'feature_contract': artifact_metadata,
    'artifacts': {path.name: {'sha256': sha256_file(path), 'bytes': path.stat().st_size} for path in artifact_paths},
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f'X_train: {split.X_train.shape}; X_test: {split.X_test.shape}')
print('Train:', split.y_train.value_counts().to_dict())
print('Final test đã khóa; TV2 không tính metric trên tập này.')
print(f'Manifest: {manifest_path}')

## 10. Minh họa SMOTE trên tập development

SMOTE chỉ được minh họa trên development; final test không thay đổi. TV3 phải đánh giá SMOTE bên trong từng fold CV thay vì cân bằng trước cross-validation.

In [ ]:
from src.features import apply_smote

X_train_smote, y_train_smote = apply_smote(split.X_train, split.y_train, random_state=2026)
smote_comparison = pd.DataFrame({
    'before': split.y_train.value_counts(),
    'after_smote': pd.Series(y_train_smote).value_counts(),
}).fillna(0).astype(int)
display(smote_comparison)
print(f'Ma trận sau SMOTE: {X_train_smote.shape}')

## Kết luận bàn giao

- Biểu đồ EDA được lưu ở `reports/figures/`.
- Pipeline chính là text-only, khớp với ứng dụng nhập review.
- Vectorizer: `models/text_tfidf_vectorizer.joblib`.
- Extractor: `models/text_feature_extractor.joblib`.
- Split khóa cho modeling: `models/train_test_features.joblib`.
- Contract/checksum/runtime: `models/artifact_manifest.json`.
- TV3 chọn model và imbalance strategy bằng CV trên train, sau đó đánh giá final test đúng một lần.